In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Konfigurace správy šumu s Estimatorem

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Existuje několik způsobů správy šumu, typicky pomocí různých technik zmírnění chyb a potlačení chyb, aby se chybám předešlo ještě před jejich vznikem. Tyto techniky obvykle způsobují předzpracovací režii. Proto je důležité dosáhnout rovnováhy mezi zdokonalením výsledků a zajištěním, že tvá úloha skončí v přiměřeném čase.

Estimator podporuje následující techniky správy šumu. Vysvětlení každé z nich najdeš v části [Techniky zmírnění a potlačení chyb](error-mitigation-and-suppression-techniques). Pokyny pro povolení těchto technik najdeš v části [Vlastní nastavení chyb](#advanced-error).

- [dynamické odpojování](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauliho twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Úroveň odolnosti
`resilience_level` určuje, jakou míru odolnosti budovat vůči chybám. Vyšší úrovně generují přesnější výsledky za cenu delší doby zpracování. Úrovně odolnosti lze použít ke konfiguraci kompromisu mezi náklady a přesností při aplikaci správy šumu na tvůj dotaz na primitivum. Správa šumu snižuje chyby (zkreslení) ve výsledcích zpracováním výstupů ze sady nebo souboru souvisejících obvodů. Míra snížení chyb závisí na použité metodě. Úroveň odolnosti abstrahuje podrobnou volbu metody správy šumu a umožňuje uživatelům uvažovat o kompromisu nákladů a přesnosti, který je vhodný pro jejich aplikaci.

Každá úroveň proto odpovídá metodě nebo metodám s rostoucí úrovní kvantové vzorkovací režie, aby sis mohl experimentovat s různými kompromisy mezi časem a přesností. Následující tabulka ukazuje, které úrovně a odpovídající metody jsou dostupné pro každé z primitiv.

<span id="resilience-table"></span>

| Úroveň odolnosti | Popis                                                                                            | Technika                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Bez zmírnění                                                                                         | Žádná                                                                      |
| 1 [Výchozí]      | Minimální náklady na zmírnění: Zmírnění chyb spojených s chybami čtení               | Twirling pro měření [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex)             |
| 2                | Střední náklady na zmírnění. Typicky snižuje zkreslení v odhadech, ale není zaručeno nulové zkreslení. | Úroveň 1 + [Extrapolace nulového šumu (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) a gate twirling

> **Info:** Úrovně odolnosti jsou momentálně ve fázi beta, takže vzorkovací režie a kvalita řešení se budou lišit obvod od obvodu. Nové funkce, pokročilé možnosti a nástroje pro správu budou vydávány průběžně. Není zaručeno, že konkrétní metody správy šumu budou aplikovány na každé úrovni odolnosti.

### Konfigurace Estimatoru s úrovněmi odolnosti
Úrovně odolnosti můžeš použít k určení technik správy šumu, nebo můžeš nastavit vlastní techniky jednotlivě, jak je popsáno v části [Vlastní nastavení chyb](#advanced-error).

> **Note:** Jakékoli možnosti, které ručně zadáš navíc k úrovni odolnosti, se aplikují navíc ke základní sadě možností definované úrovní odolnosti. Proto bys v principu mohl nastavit úroveň odolnosti na 1, ale poté vypnout zmírnění chyb měření, i když to není doporučeno.
> 
> Například nastavení úrovně odolnosti na 0 vypne `zne_mitigation`, ale `estimator.options.resilience.zne_mitigation = True` tuto hodnotu přepíše.

### Příklad
Následující kód povoluje ZNE, TREX a gate twirling nastavením `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Vlastní nastavení správy šumu
Jednotlivé metody správy šumu můžeš zapínat a vypínat pomocí [možností Estimatoru](/guides/estimator-options).

> **Note:** Ne všechny možnosti fungují dohromady se všemi typy obvodů. Podrobnosti najdeš v [tabulce kompatibility funkcí](estimator-options#options-compatibility-table).

### Příklad

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: "
    f"{estimator.options.twirling.enable_gates}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>